# Overpass
[info overpass link](https://wiki.openstreetmap.org/wiki/Overpass_turbo)


In [ ]:
import requests, pandas as pd

query = """
[out:json][timeout:120];
nwr["shop"~"^(supermarket|convenience)$"](10.88,-74.90,11.06,-74.74);
out center;
"""

headers = {
    "User-Agent": "barranquilla-retail-whitespace/1.0 (portfolio project)",
    "Accept": "application/json",
}
servers = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

data = None
for url in servers:
    r = requests.post(url, data={"data": query}, headers=headers, timeout=180)
    print(url, r.status_code)
    if r.status_code == 200:
        data = r.json()
        break
    print(r.text[:300])  # shows the server's reason if it fails

rows = []
for e in data["elements"]:
    t = e.get("tags", {})
    rows.append({
        "osm_id": f"{e['type']}/{e['id']}",
        "lat": e.get("lat") or e.get("center", {}).get("lat"),
        "lon": e.get("lon") or e.get("center", {}).get("lon"),
        "name": t.get("name"),
        "brand": t.get("brand"),
        "shop": t.get("shop"),
    })

df = pd.DataFrame(rows)
print(len(df), "stores")
df.head()

https://overpass-api.de/api/interpreter 200
500 stores


,osm_id,lat,lon,name,brand,shop
0,node/836805950,10.965520,-74.833672,Tienda Los Paisas,None,convenience
1,node/922818929,10.904249,-74.789296,None,None,convenience
2,node/1131254873,11.037947,-74.825471,Donde José,None,convenience
3,node/1131254907,11.038989,-74.824316,El Baraton,None,convenience
4,node/2154764347,10.939099,-74.772198,El York,None,convenience


In [ ]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
PROJECT = "projectgiscol"
client = bigquery.Client(project=PROJECT)

# 1. Create the dataset, in the US location
ds = bigquery.Dataset(f"{PROJECT}.geo_retail")
ds.location = "US"
client.create_dataset(ds, exists_ok=True)

# 2. Loading the stores into geo_retail.osm_raw
df["lat"] = df["lat"].astype(float)
df["lon"] = df["lon"].astype(float)
job = client.load_table_from_dataframe(
    df, f"{PROJECT}.geo_retail.osm_raw",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print("osm_raw rows:", client.get_table(f"{PROJECT}.geo_retail.osm_raw").num_rows)

# 3. Creating the clean table from Step 1
client.query("""
CREATE OR REPLACE TABLE geo_retail.stores_osm AS
SELECT
  CAST(osm_id AS STRING) AS store_id,
  name,
  brand,
  shop AS category,
  ST_GEOGPOINT(lon, lat) AS geom
FROM geo_retail.osm_raw
WHERE lat IS NOT NULL AND lon IS NOT NULL
""").result()
print("stores_osm rows:", client.get_table(f"{PROJECT}.geo_retail.stores_osm").num_rows)



osm_raw rows: 500
stores_osm rows: 500


# Overture maps
[Website Link](https://overturemaps.org/)

In [7]:
!pip install -q overturemaps
!overturemaps download --bbox=-74.90,10.88,-74.74,11.06 -f geoparquet --type=place -o barranquilla_places.parquet
!ls -lh barranquilla_places.parquet

Downloading: 23459 rows [00:01, 18816.21 rows/s]
State saved to barranquilla_places.parquet.state
-rw-r--r-- 1 root root 4.4M Oct 12 23:42 barranquilla_places.parquet


In [9]:
import geopandas as gpd, pandas as pd
gdf = gpd.read_parquet("barranquilla_places.parquet")
print(len(gdf), "places")

23459 places


In [12]:
import pandas as pd
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
PROJECT = "projectgiscol"
client = bigquery.Client(project=PROJECT)

# Helper to read nested fields
def get(d, *keys):
    for k in keys:
        if d is None:
            return None
        d = d.get(k) if isinstance(d, dict) else None
    return d

gdf["name"] = gdf["names"].apply(lambda n: get(n, "primary"))
gdf["brand_name"] = gdf["brand"].apply(lambda b: get(b, "names", "primary"))
gdf["category"] = gdf["basic_category"]
gdf["tax_primary"] = gdf["taxonomy"].apply(lambda t: get(t, "primary"))

# Grocery categories + chains filed under other categories
pattern = "grocer|supermarket|convenience|discount"
base = (gdf["category"].fillna("").str.contains(pattern, case=False)
        | gdf["tax_primary"].fillna("").str.contains(pattern, case=False))
chain_name = gdf["name"].str.contains(r"\bD1\b|\bAra\b|Ol[ií]mpica|Isimo", case=False, na=False)
chain_cats = ["food_and_beverage_store", "department_store", "discount_store", "shopping"]
stores = gdf[base | (chain_name & gdf["category"].isin(chain_cats))]
print(len(stores), "stores before confidence filter")
print(stores["confidence"].describe())

# Drop low-confidence places
stores = stores[stores["confidence"] >= 0.5]
print(len(stores), "stores after confidence filter")

df_ov = pd.DataFrame({
    "store_id": stores["id"], "name": stores["name"], "brand": stores["brand_name"],
    "category": stores["category"], "confidence": stores["confidence"],
    "lat": stores.geometry.y, "lon": stores.geometry.x,
})

cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(df_ov, f"{PROJECT}.geo_retail.overture_raw", job_config=cfg).result()
client.query("""
CREATE OR REPLACE TABLE geo_retail.stores_overture AS
SELECT store_id, name, brand, category, ST_GEOGPOINT(lon, lat) AS geom
FROM geo_retail.overture_raw
""").result()
print("stores_overture rows:", client.get_table(f"{PROJECT}.geo_retail.stores_overture").num_rows)

380 stores before confidence filter
count    380.000000
mean       0.545849
std        0.245150
min        0.033977
25%        0.329455
50%        0.542934
75%        0.770000
max        0.998299
Name: confidence, dtype: float64
216 stores after confidence filter
stores_overture rows: 216


# miciudad.barranquilla
[mi.ciudadbarranquilla - Gis link](https://miciudad.barranquilla.gov.co/gis/rest/services)

In [15]:
# ===== 1. Libraries and BigQuery connection =====
import geopandas as gpd, pandas as pd
from google.colab import auth, files
from google.cloud import bigquery

auth.authenticate_user()
PROJECT = "projectgiscol"
client = bigquery.Client(project=PROJECT)
cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

# ===== 2. Upload the 5 GeoJSON files (select all of them in the picker) =====
uploaded = files.upload()
estr = pd.concat([gpd.read_file(name) for name in uploaded], ignore_index=True)
print(len(estr), "blocks (expected 8287)")
print(estr["estratificacion"].value_counts())

# ===== 3. One center point per block + clean estrato number =====
pts = estr.to_crs(9377).geometry.centroid.to_crs(4326)
blocks_estrato = pd.DataFrame({
    "barrio": estr["nombre_barrio"],
    "localidad": estr["localidad"],
    "estrato": estr["estratificacion"].astype(str).str.extract(r"(\d)")[0].astype(float),  # "No aplica" -> null
    "lat": pts.y,
    "lon": pts.x,
})
print(blocks_estrato.head())

# ===== 4. Load to BigQuery =====
client.load_table_from_dataframe(
    blocks_estrato, f"{PROJECT}.geo_retail.blocks_estrato", job_config=cfg
).result()
print("blocks_estrato loaded:", len(blocks_estrato))

Saving estrato_0.geojson to estrato_0.geojson
Saving estrato_1.geojson to estrato_1.geojson
Saving estrato_2.geojson to estrato_2.geojson
Saving estrato_3.geojson to estrato_3.geojson
Saving estrato_4.geojson to estrato_4.geojson
8287 blocks (expected 8287)
estratificacion
No aplica    2098
1            1831
2            1718
3             955
4             926
5             388
6             364
Name: count, dtype: int64
          barrio localidad  estrato        lat        lon
0     Bellavista        02      4.0  11.001068 -74.794444
1       El Prado        02      4.0  10.995857 -74.792354
2  San Francisco        02      3.0  11.003327 -74.793663
3      Santa Ana        02      4.0  10.999647 -74.791589
4  San Francisco        02      3.0  11.002065 -74.792299
blocks_estrato loaded: 8287
